## Example 1: Docker Worker Configuration

In [ ]:
from prefect.workers.base import BaseWorker
from prefect_docker import DockerWorker
from prefect_docker.worker import DockerWorkerJobConfiguration

# Example 1: Docker Worker Configuration. Nothing here needs a Docker daemon.
print("Example 1 - Docker Worker Configuration")

print(f"DockerWorker class available:")
print(f"  Module: {DockerWorker.__module__}")
print(f"  Worker type: {DockerWorker.type}")
assert issubclass(DockerWorker, BaseWorker)
assert DockerWorker.type == "docker", DockerWorker.type
assert DockerWorker.job_configuration is DockerWorkerJobConfiguration

expected_methods = ["run", "setup", "teardown", "kill_infrastructure", "get_default_base_job_template"]
missing = [m for m in expected_methods if not hasattr(DockerWorker, m)]
assert not missing, f"DockerWorker is missing {missing}"
print(f"  Worker methods present: {', '.join(expected_methods)}")

# The default base job template maps each job setting to a work-pool variable.
template = DockerWorker.get_default_base_job_template()
job_config = template["job_configuration"]
variables = template["variables"]["properties"]
print(f"\nBase job template: {len(job_config)} job settings, {len(variables)} variables")
for key in ("image", "env", "volumes", "auto_remove", "mem_limit"):
    assert key in variables, key
    assert job_config[key] == "{{ " + key + " }}", (key, job_config[key])

# A job configuration built directly keeps the values it is given.
config = DockerWorkerJobConfiguration(
    image="python:3.12-slim",
    env={"APP_ENV": "test"},
    labels={"team": "data"},
    mem_limit="512m",
    auto_remove=True,
)
print(f"\nJob configuration: image={config.image}, mem_limit={config.mem_limit}, auto_remove={config.auto_remove}")
assert config.image == "python:3.12-slim"
assert config.env == {"APP_ENV": "test"}
assert config.labels == {"team": "data"}
assert config.mem_limit == "512m"
assert config.auto_remove is True

print("Example 1 completed successfully")


## Example 2: Docker Host Connection Management

In [ ]:
from prefect_docker import DockerHost
from prefect import flow, task, get_run_logger

# Example 2: Docker host connection settings. The blocks only hold settings; no daemon is contacted.
print("Example 2 - Docker Host Connection Management")

# Local Docker daemon connection
local_docker = DockerHost(
    base_url="unix://var/run/docker.sock",
    max_pool_size=10
)
print(f"Local Docker Host:")
print(f"  Base URL: {local_docker.base_url}")
print(f"  Max pool size: {local_docker.max_pool_size}")
assert local_docker.base_url == "unix://var/run/docker.sock"
assert local_docker.max_pool_size == 10

# Remote Docker daemon connection (TCP)
remote_docker = DockerHost(
    base_url="tcp://192.0.2.10:2376",
    max_pool_size=5,
    timeout=30,
)
print(f"\nRemote Docker Host:")
print(f"  Base URL: {remote_docker.base_url}")
print(f"  Max pool size: {remote_docker.max_pool_size}")
assert remote_docker.base_url == "tcp://192.0.2.10:2376"
assert remote_docker.max_pool_size == 5
assert remote_docker.timeout == 30

# Docker host with an API version and extra client keyword arguments
custom_docker = DockerHost(
    base_url="unix://var/run/docker.sock",
    version="1.45",
    max_pool_size=20,
    client_kwargs={"use_ssh_client": False}
)
print(f"\nCustom Docker Host:")
print(f"  Base URL: {custom_docker.base_url}")
print(f"  API version: {custom_docker.version}")
print(f"  Client kwargs: {custom_docker.client_kwargs}")
assert custom_docker.version == "1.45"
assert custom_docker.max_pool_size == 20
assert custom_docker.client_kwargs == {"use_ssh_client": False}

# Several hosts
docker_hosts = {
    "primary": DockerHost(base_url="unix://var/run/docker.sock", max_pool_size=15),
    "secondary": DockerHost(base_url="tcp://docker-host-2:2376", max_pool_size=10),
    "backup": DockerHost(base_url="tcp://docker-host-3:2376", max_pool_size=5),
}

@task(name="describe-host")
def describe_docker_host(docker_host):
    """Summarise a host's connection settings"""
    logger = get_run_logger()
    logger.info("Describing %s", docker_host.base_url)
    return {
        "protocol": docker_host.base_url.split("://", 1)[0],
        "pool_size": docker_host.max_pool_size,
    }

@flow(name="docker-host-management")
def docker_host_management_flow():
    """Collect the settings of every configured host"""
    return {name: describe_docker_host(host) for name, host in docker_hosts.items()}

summary = docker_host_management_flow()
print(f"\nDocker host management flow result: {summary}")
assert summary == {
    "primary": {"protocol": "unix", "pool_size": 15},
    "secondary": {"protocol": "tcp", "pool_size": 10},
    "backup": {"protocol": "tcp", "pool_size": 5},
}, summary

total_pool_capacity = sum(host["pool_size"] for host in summary.values())
print(f"Total pool capacity: {total_pool_capacity}")
assert total_pool_capacity == 30

print("Example 2 completed successfully")
